Run model for at least 5 years and then 1 year each after this to see when SST and air temperature stabilises

In [1]:
using Pkg
Pkg.status()

Status `~/Documents/rotation_project/Project.toml`
  [9e226e20] SpeedyWeather v0.18.1


In [2]:
using SpeedyWeather

In [3]:
spectral_grid = SpectralGrid()

SpectralGrid{Spectrum{...}, OctahedralGaussianGrid{...}}
├ Number format: Float32
├ Spectral:      T31 LowerTriangularMatrix
├ Grid:          48-ring OctahedralGaussianGrid, 3168 grid points
├ Resolution:    3.61°, 401km (at 6371km radius)
├ Vertical:      8-layer atmosphere
└ Architecture:  CPU using Array

In [4]:
at = 80
flux = 1.25 * 1365
outdir_at = "/Users/woodh/Documents/rotation_project"
fname = joinpath(outdir_at, "output.nc") 
output = NetCDFOutput(spectral_grid; filename=fname)

NetCDFOutput{Field{Float32, 1, Vector{Float32}, FullGaussianGrid{CPU{KernelAbstractions.CPU}, Vector{UnitRange{Int64}}, Vector{Int64}}}}
├ status: inactive/uninitialized
├ write restart file: true (if active)
├ interpolator: AnvilInterpolator{Float32, RingGrids.GridGeometry{OctahedralGaussianGrid{CPU{KernelAbstractions.CPU}, Vector{UnitRange{Int64}}, Vector{Int64}}, Vector{Float32}, Vector{Int64}, Vector{UnitRange{Int64}}}, RingGrids.AnvilLocator{Vector{Float32}, Vector{Int64}}}
├ path: /Users/woodh/Documents/rotation_project/output.nc (overwrite=false)
├ frequency: 21600 seconds
└┐ variables:
 ├ v: meridional wind [m/s]
 ├ u: zonal wind [m/s]
 └ vor: relative vorticity [s^-1]

In [5]:
add!(output, SpeedyWeather.DivergenceOutput()) 
add!(output, SpeedyWeather.VorticityOutput())
add!(output, SpeedyWeather.TemperatureOutput())
add!(output, SpeedyWeather.SurfaceShortwaveDownOutput())
add!(output, SpeedyWeather.SoilMoistureOutput())
add!(output, SpeedyWeather.SoilTemperatureOutput())
add!(output, SpeedyWeather.SurfaceTemperatureOutput())
add!(output, SpeedyWeather.LandSeaMaskOutput())

NetCDFOutput{Field{Float32, 1, Vector{Float32}, FullGaussianGrid{CPU{KernelAbstractions.CPU}, Vector{UnitRange{Int64}}, Vector{Int64}}}}
├ status: inactive/uninitialized
├ write restart file: true (if active)
├ interpolator: AnvilInterpolator{Float32, RingGrids.GridGeometry{OctahedralGaussianGrid{CPU{KernelAbstractions.CPU}, Vector{UnitRange{Int64}}, Vector{Int64}}, Vector{Float32}, Vector{Int64}, Vector{UnitRange{Int64}}}, RingGrids.AnvilLocator{Vector{Float32}, Vector{Int64}}}
├ path: /Users/woodh/Documents/rotation_project/output.nc (overwrite=false)
├ frequency: 21600 seconds
└┐ variables:
 ├ lsm: land-sea mask (1=land, 0=sea) [1]
 ├ st: soil temperature [degC]
 ├ tsurf: Surface air temperature [degC]
 ├ sm: soil moisture [1]
 ├ v: meridional wind [m/s]
 ├ temp: temperature [degC]
 ├ div: divergence [s^-1]
 ├ u: zonal wind [m/s]
 ├ srd: Surface shortwave radiation down [W/m^2]
 └ vor: relative vorticity [s^-1]

In [ ]:
# p = Earth(spectral_grid, solar_constant=flux)
p = Earth(spectral_grid)
model = PrimitiveWetModel(spectral_grid; output=output, planet=p)

add!(model, SpeedyWeather.SnowDepthOutput())
add!(model, SpeedyWeather.SnowMeltOutput())
add!(model, SpeedyWeather.RadiationOutput())
add!(model, SpeedyWeather.OceanOutput())
add!(model, SpeedyWeather.PrecipitationOutput())
add!(model, SpeedyWeather.HumidityOutput())

In [ ]:
sim = initialize!(model)

In [ ]:
run!(sim, period=Year(5), output=true)